#  Multiple Outputs, Softmax & Argmax

## Multiple Outputs / Classification
- A neural network can produce multiple outputs for each input sample.
- Used `nn.Linear(6, 3)` to produce **3 logits per sample**.
- If input shape is `(5, 6)`:
  - `5` = number of samples
  - `6` = number of input features
  - output shape `(5, 3)` = 3 class scores for each sample.
- These raw outputs are called **logits**.
- One tensor contains all class outputs; each output does not need to be returned separately.

## Logits → Probabilities
- Logits are raw scores, **not probabilities**.
- **Softmax** converts logits into probabilities.
- Probabilities across the classes sum to approximately `1`.
- Higher logit → higher probability.

## Argmax
- **Argmax** selects the index of the largest probability.
- That index becomes the predicted class.
- Example:
  - `[0.20, 0.65, 0.15]`
  - prediction → class `1`.

## Softmax Derivative
- Tested Softmax independently using a small random tensor.
- No second neural network was necessary.
- Used a **Jacobian** to examine the derivatives of Softmax.
- For 2 inputs → 2 Softmax outputs → a **2 × 2 Jacobian**.
- Each entry represents how one output changes with respect to one input.
- Diagonal terms → effect of a logit on its corresponding probability.
- Off-diagonal terms → effect of one logit on another class's probability.
- Negative off-diagonal values make sense because increasing one class's probability affects the probabilities of the other classes.



In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd.functional import jacobian

In [10]:
X = torch.randn(5,4)
y = torch.tensor([0,1,2])

In [7]:
class NN(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(4,6)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(6,3)
  def forward(self, x):
    print(x.shape)
    x = self.fc1(x)
    print(x.shape)
    x = self.relu(x)
    logits = self.fc2(x)
    print(logits.shape)
    print(logits)
    return logits

In [16]:
model = NN()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
logits = model(X)
prob = F.softmax(logits, dim = 1)
print(prob)
torch.argmax(prob, dim = 1)


torch.Size([5, 4])
torch.Size([5, 6])
torch.Size([5, 3])
tensor([[ 0.0270, -0.8366,  0.3304],
        [ 0.3587,  0.1935, -0.2704],
        [ 0.0063, -0.1989, -0.2414],
        [ 0.1817, -0.5858,  0.1425],
        [ 0.1124, -0.3700, -0.0126]], grad_fn=<AddmmBackward0>)
tensor([[0.3602, 0.1519, 0.4879],
        [0.4200, 0.3561, 0.2239],
        [0.3853, 0.3139, 0.3008],
        [0.4122, 0.1914, 0.3964],
        [0.4000, 0.2469, 0.3530]], grad_fn=<SoftmaxBackward0>)


tensor([2, 0, 0, 0, 0])

In [24]:
x = torch.randn(1,2)
print(x)
jacob = jacobian(F.softmax, x)
print(jacob)

tensor([[ 0.1856, -0.4145]])
tensor([[[[ 0.2288, -0.2288]],

         [[-0.2288,  0.2288]]]])


/usr/local/lib/python3.12/dist-packages/torch/autograd/functional.py:700: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  outputs = func(*inputs)
